# 1D Heat Equation (Implicit) - Analytical to Numerical Bridge

This notebook demonstrates the mathematical derivation, numerical implementation, and performance benchmarking of the 1D Heat Equation using an **Implicit Backward Euler** scheme.

## 1. Physics Background

The 1D Heat Equation models the distribution of temperature $T(x, t)$ over time. In this notebook, we focus on the **Implicit** numerical implementation, which allows for larger time steps than explicit methods.

## 2. Detailed Analytical Trace (Sympy)

We use **Separation of Variables** to derive the exact solution.

In [ ]:
import sympy as sp
from IPython.display import display, Math

sp.init_printing()

# 1. Symbolic Definition
x, t, alpha, L = sp.symbols('x t alpha L', real=True, positive=True)
T = sp.Function('T')(x, t)

# Assume T(x, t) = X(x) * Phi(t)
X = sp.Function('X')(x)
Phi = sp.Function('Phi')(t)
lam = sp.symbols('lambda', real=True, positive=True)

ode_spatial = sp.Eq(X.diff(x, x) + lam**2 * X, 0)
ode_temporal = sp.Eq(Phi.diff(t) + alpha * lam**2 * Phi, 0)

display(Math(f"\\text{{Spatial ODE: }} {sp.latex(ode_spatial)}"))
display(Math(f"\\text{{Temporal ODE: }} {sp.latex(ode_temporal)}"))

sol_x = sp.dsolve(ode_spatial, X)
sol_phi = sp.dsolve(ode_temporal, Phi)
display(Math(f"X(x) = {sp.latex(sol_x.rhs)}"))
display(Math(f"\\Phi(t) = {sp.latex(sol_phi.rhs)}"))

### Eigenvalues and Series Solution

Applying $T(0,t)=0$ and $T(L,t)=0$ yields $\lambda_n = \frac{n\pi}{L}$. The general solution is a Fourier sine series:
$$T(x, t) = \sum_{n=1}^{\infty} A_n \sin\left(\frac{n\pi x}{L}\right) e^{-\alpha (\frac{n\pi}{L})^2 t}$$

## 3. Implicit Numerical Implementation

The Backward Euler scheme is given by:
$$\frac{T_i^{n+1} - T_i^n}{\Delta t} = \alpha \frac{T_{i+1}^{n+1} - 2T_i^{n+1} + T_{i-1}^{n+1}}{\Delta x^2}$$

This leads to a **tridiagonal system of linear equations** that must be solved at each time step.

In [ ]:
import numpy as np
from scipy.linalg import solve_banded
import matplotlib.pyplot as plt

nx = 101
L = 1.0
x_vals = np.linspace(0, L, nx)
dx = L / (nx - 1)
alpha_val = 0.05 
dt = 0.01 # Large time step (unstable for explicit, stable for implicit)
t_final = 0.2
steps = int(t_final / dt)

u0 = np.exp(-100 * (x_vals - 0.5)**2)
u0[0] = u0[-1] = 0.0

def solve_implicit(u, alpha, dx, dt, steps):
    u_curr = u.copy()
    r = alpha * dt / dx**2
    
    # Tridiagonal Matrix: (1+2r) on diag, -r on sub and super diag
    diag = (1 + 2*r) * np.ones(nx-2)
    off_diag = -r * np.ones(nx-3)
    
    # solve_banded format: (upper, middle, lower)
    ab = np.zeros((3, nx-2))
    ab[0, 1:] = off_diag
    ab[1, :] = diag
    ab[2, :-1] = off_diag
    
    for _ in range(steps):
        rhs = u_curr[1:-1]
        u_curr[1:-1] = solve_banded((1, 1), ab, rhs)
    return u_curr

def evaluate_analytical(x_vals, t_v, alpha_v, L_v, u_init, n_terms=50):
    T_final = np.zeros_like(x_vals)
    dx_sim = L_v / (len(x_vals)-1)
    for n_val in range(1, n_terms + 1):
        kn = n_val * np.pi / L_v
        An = (2.0/L_v) * np.sum(u_init * np.sin(kn * x_vals)) * dx_sim
        T_final += An * np.sin(kn * x_vals) * np.exp(-alpha_v * kn**2 * t_v)
    return T_final

u_implicit = solve_implicit(u0, alpha_val, dx, dt, steps)
u_exact = evaluate_analytical(x_vals, t_final, alpha_val, L, u0)

plt.figure(figsize=(10, 6))
plt.plot(x_vals, u0, 'k:', label='Initial condition')
plt.plot(x_vals, u_implicit, 'b-', label='Numerical (Implicit)')
plt.plot(x_vals, u_exact, 'go', markersize=4, alpha=0.5, label='Analytical Trace')
plt.title("Implicit Solver Verification against Analytical Series")
plt.legend()
plt.grid(True)
plt.show()